# Clasificación de imágenes con CNN (`CNNriesgo`)

Este notebook entrena una **red neuronal convolucional (CNN)** para clasificar
imágenes según la **carpeta** en la que están (una carpeta = una clase).

**Flujo general**

1. Leer imágenes desde subdirectorios
2. Crear etiquetas numéricas y nombres de clase
3. Dividir en train / test / validación
4. Normalizar píxeles y codificar etiquetas (one-hot)
5. Definir, compilar y entrenar la CNN
6. Evaluar, graficar curvas y predecir sobre imágenes nuevas

> Teoría detallada: apuntes *Redes Neuronales Convolucionales (CNN)* en
> `introduccion_inteligencia_artificial.org`.


## 0. Librerías

| Librería | Para qué |
|----------|----------|
| `numpy` | Arreglos de imágenes y etiquetas |
| `os`, `re` | Recorrer carpetas y filtrar extensiones |
| `matplotlib` | Ver imágenes y curvas de entrenamiento |
| `sklearn` | `train_test_split` y `classification_report` |
| `tensorflow` / `keras` | Capas CNN, entrenamiento e inferencia |


In [ ]:
import numpy as np
import os
import re
import cv2
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import keras
import tensorflow as tf
from keras.utils import to_categorical
from keras.models import Sequential, load_model
from keras.layers import (
    Conv2D, MaxPooling2D, Dense, Dropout, Flatten, LeakyReLU
)
from skimage.transform import resize

# Tamaño fijo que espera el modelo (alto, ancho, canales RGB)
# Usamos 28x28 porque las caras recortadas son cuadradas
IMG_H, IMG_W, IMG_C = 28, 28, 3


## 1. Leer imágenes

Se recorre un directorio raíz. **Cada subcarpeta es una clase**
(ej. `inundacion/`, `incendios/`, `tornados/`).

Se aceptan: `.jpg`, `.jpeg`, `.png`, `.bmp`, `.tiff`.

Solo se guardan imágenes con 3 canales (RGB). Ajusta `DATASET_DIR` a tu ruta.


In [ ]:
# Apuntamos a la carpeta de caras de nuestro proyecto
DATASET_DIR = './haarcascade_dataset'

imgpath = os.path.join(DATASET_DIR, '')
images = []
directories = []
dircount = []
prev_root = ''
cant = 0

print('Leyendo imágenes de', imgpath)

for root, dirnames, filenames in os.walk(imgpath):
    # Ignoramos carpetas del sistema como .git
    if '.git' in root:
        continue
    for filename in filenames:
        if re.search(r'\.(jpg|jpeg|png|bmp|tiff)$', filename, re.IGNORECASE):
            cant += 1
            filepath = os.path.join(root, filename)
            image = plt.imread(filepath)
            if len(image.shape) == 3:
                # Si trae 4 canales (RGBA), dejamos solo 3 (RGB)
                if image.shape[2] == 4:
                    image = image[:, :, :3]
                # REDIMENSIONAR: obligatorio porque cada cara tiene tamaños distintos
                image_resized = resize(image, (IMG_H, IMG_W), preserve_range=True, anti_aliasing=True)
                images.append(image_resized)
            if prev_root != root:
                prev_root = root
                directories.append(root)
                dircount.append(cant)
                cant = 0

dircount.append(cant)
dircount = dircount[1:]
dircount[0] = dircount[0] + 1

print('Directorios leídos:', len(directories))
print('Imágenes en cada directorio:', dircount)
print('Total de imágenes:', sum(dircount))


## 2. Etiquetas numéricas

Si la primera carpeta tiene 100 imágenes → etiquetas `0, 0, …, 0`.
La segunda → `1, 1, …`, y así sucesivamente.

Eso es lo que la red aprenderá a predecir.


In [ ]:
labels = []
indice = 0
for cantidad in dircount:
    for _ in range(cantidad):
        labels.append(indice)
    indice += 1
print('Cantidad de etiquetas creadas:', len(labels))


## 3. Nombres de clase

Se guarda el **nombre de la carpeta** (último segmento de la ruta) en `sriesgos`.
Sirve para imprimir predicciones legibles en lugar de solo el número `0`, `1`, `2`…


In [ ]:
sriesgos = []
for indice, directorio in enumerate(directories):
    name = directorio.split(os.sep)
    clase = name[-1]
    print(indice, clase)
    sriesgos.append(clase)


## 4. De listas a NumPy

- `X`: tensor de imágenes `(N, alto, ancho, 3)`
- `y`: vector de etiquetas enteras `(N,)`
- `nClasses`: número de carpetas / clases de salida


In [ ]:
y = np.array(labels)
X = np.array(images, dtype=np.uint8)

classes = np.unique(y)
nClasses = len(classes)
print('Número de clases (salidas):', nClasses)
print('Clases:', classes)
print('Forma de X:', X.shape)


## 5. Train / test (80% / 20%)

- **Train**: aprende los pesos.
- **Test**: se reserva hasta el final para medir qué tan bien generaliza
  (no se usa para ajustar hiperparámetros en este notebook).


In [ ]:
train_X, test_X, train_Y, test_Y = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train:', train_X.shape, train_Y.shape)
print('Test :', test_X.shape, test_Y.shape)


## 6. Vista rápida de una imagen

Comprueba que se leyeron bien y que la etiqueta tiene sentido.


In [ ]:
plt.figure(figsize=[6, 3])
plt.subplot(121)
plt.imshow(train_X[0])
plt.title('Train — clase: {}'.format(sriesgos[train_Y[0]]))
plt.subplot(122)
plt.imshow(test_X[0])
plt.title('Test — clase: {}'.format(sriesgos[test_Y[0]]))
plt.show()


## 7. Normalización

Los píxeles suelen ir de **0 a 255**. Dividir entre 255 los deja en **[0, 1]**,
lo que ayuda a que el optimizador converja de forma más estable.

También convertimos a `float32` (tipo que espera Keras).


In [ ]:
train_X = train_X.astype('float32') / 255.0
test_X = test_X.astype('float32') / 255.0
plt.imshow(test_X[0])
plt.title('Ejemplo normalizado (valores 0–1)')
plt.show()


## 8. One-hot encoding

La pérdida `categorical_crossentropy` espera etiquetas así:

- entero `2` → vector `[0, 0, 1]` (si hay 3 clases)

Cada posición es la probabilidad “ideal” de esa clase.


In [ ]:
train_Y_one_hot = to_categorical(train_Y)
test_Y_one_hot = to_categorical(test_Y)
print('Etiqueta original:', train_Y[0])
print('One-hot:', train_Y_one_hot[0])


## 9. Validación (desde el train)

Del 80% de entrenamiento separamos otro 20% como **validación**:
sirve para vigilar sobreajuste *durante* el `fit`, sin tocar el test final.


In [ ]:
train_X, valid_X, train_label, valid_label = train_test_split(
    train_X, train_Y_one_hot, test_size=0.2, random_state=13
)
print(train_X.shape, valid_X.shape, train_label.shape, valid_label.shape)


## 10. Hiperparámetros

| Variable | Significado |
|----------|-------------|
| `INIT_LR` | Learning rate inicial (tamaño de paso) |
| `epochs` | Cuántas veces se recorre el conjunto de entrenamiento |
| `batch_size` | Cuántas imágenes se usan en cada actualización de pesos |


In [ ]:
INIT_LR = 1e-3      # 0.001
epochs = 20
batch_size = 64


## 11. Definir la CNN

Capa a capa:

1. **Conv2D(32, 3×3)**: 32 filtros que buscan patrones locales en la imagen.
2. **LeakyReLU**: activación no lineal (deja pasar un poco los negativos).
3. **MaxPooling2D**: reduce alto/ancho (se queda con el máximo de cada 2×2).
4. **Dropout(0.5)**: apaga neuronas al azar en entrenamiento → menos sobreajuste.
5. **Flatten**: convierte el mapa de características en un vector.
6. **Dense(32)**: combina lo detectado (como un MLP pequeño).
7. **Dense(nClasses, softmax)**: una probabilidad por clase.

`input_shape=(21, 28, 3)` debe coincidir con el tamaño de tus imágenes.


In [ ]:
# ==============================================================================
# DEFINICIÓN DE LA RED NEURONAL CONVOLUCIONAL (CNN) - EXPLICADO FOR DUMMIES
# ==============================================================================
riesgo_model = Sequential([
    # --- BLOQUE 1: Detecta rasgos básicos (líneas, sombras, bordes) ---
    Conv2D(32, kernel_size=(3, 3), activation='linear', padding='same',
           input_shape=(IMG_H, IMG_W, IMG_C)),  # 32 filtros que escanean la imagen
    LeakyReLU(alpha=0.1),                       # Enciende o apaga neuronas
    MaxPooling2D((2, 2), padding='same'),       # Reduce la imagen a la mitad (se queda con lo más importante)
    Dropout(0.5),                               # Apaga el 50% de neuronas al azar para evitar memorización (sobreajuste)

    # --- BLOQUE 2 (CAPA AGREGADA): Detecta rasgos más complejos (ojos, nariz, boca) ---
    Conv2D(64, kernel_size=(3, 3), activation='linear', padding='same'),  # Subimos a 64 filtros
    LeakyReLU(alpha=0.1),                       # Activación
    MaxPooling2D((2, 2), padding='same'),       # Vuelve a reducir el tamaño
    Dropout(0.5),                               # Vuelve a evitar memorización

    # --- CLASIFICADOR FINAL (Capas densas / totalmente conectadas) ---
    Flatten(),                                  # Aplana la imagen 2D a una sola fila de números
    Dense(32, activation='linear'),             # Neuronas que combinan todo lo detectado
    LeakyReLU(alpha=0.1),
    Dropout(0.5),
    Dense(nClasses, activation='softmax'),      # Capa final: calcula la probabilidad para cada persona
])

riesgo_model.summary()


## 12. Compilar

- **loss** `categorical_crossentropy`: mide qué tan lejos están las
  probabilidades predichas de la etiqueta one-hot.
- **optimizer** SGD: baja el error moviendo pesos en dirección opuesta al gradiente.
- **metrics** `accuracy`: porcentaje de aciertos (para monitorear, no es la pérdida).


In [ ]:
riesgo_model.compile(
    loss=keras.losses.categorical_crossentropy,
    optimizer=tf.keras.optimizers.SGD(learning_rate=INIT_LR, decay=INIT_LR / 100),
    metrics=['accuracy'],
)


## 13. Entrenar

`fit` ajusta los pesos. `validation_data` muestra en cada época cómo va
en datos que **no** se usan para actualizar pesos (señal de sobreajuste).


In [ ]:
riesgo_train = riesgo_model.fit(
    train_X, train_label,
    batch_size=batch_size,
    epochs=epochs,
    verbose=1,
    validation_data=(valid_X, valid_label),
)


## 14. Guardar el modelo

Así puedes cargarlo después sin volver a entrenar (`load_model`).


In [ ]:
MODEL_PATH = 'modelo_caras.h5'
riesgo_model.save(MODEL_PATH)
print('Modelo guardado en', MODEL_PATH)


## 15. Evaluar en el test

Pérdida y accuracy sobre el conjunto que **nunca** entró al entrenamiento
ni a la validación.


In [ ]:
test_eval = riesgo_model.evaluate(test_X, test_Y_one_hot, verbose=1)
print('Test loss:', test_eval[0])
print('Test accuracy:', test_eval[1])


## 16. Curvas de aprendizaje

- Si **train** sube y **val** se estanca o baja → sobreajuste.
- Si ambas suben juntas → buen aprendizaje.
- Si ambas se quedan bajas → subajuste (pocas épocas, red débil o datos difíciles).


In [ ]:
acc = riesgo_train.history['accuracy']
val_acc = riesgo_train.history['val_accuracy']
loss = riesgo_train.history['loss']
val_loss = riesgo_train.history['val_loss']
ep = range(len(acc))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(ep, acc, 'bo-', label='Train accuracy')
plt.plot(ep, val_acc, 'b-', label='Val accuracy')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(ep, loss, 'ro-', label='Train loss')
plt.plot(ep, val_loss, 'r-', label='Val loss')
plt.title('Loss')
plt.legend()
plt.tight_layout()
plt.show()


## 17. Predicciones en el test

`predict` devuelve un vector de probabilidades por imagen.
Tomamos el índice del máximo (`argmax`) → clase entera.


In [ ]:
probs = riesgo_model.predict(test_X)
predicted_classes = np.argmax(probs, axis=1)
print(predicted_classes.shape, test_Y.shape)


## 18. Ejemplos bien / mal clasificados

Título: `predicha, real` (nombres de carpeta).


In [ ]:
correct = np.where(predicted_classes == test_Y)[0]
print('Aciertos:', len(correct))
plt.figure(figsize=(8, 8))
for i, idx in enumerate(correct[:9]):
    plt.subplot(3, 3, i + 1)
    plt.imshow(test_X[idx])
    plt.title('{}, {}'.format(sriesgos[predicted_classes[idx]], sriesgos[test_Y[idx]]))
    plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
incorrect = np.where(predicted_classes != test_Y)[0]
print('Errores:', len(incorrect))
plt.figure(figsize=(8, 8))
for i, idx in enumerate(incorrect[:9]):
    plt.subplot(3, 3, i + 1)
    plt.imshow(test_X[idx])
    plt.title('{}, {}'.format(sriesgos[predicted_classes[idx]], sriesgos[test_Y[idx]]))
    plt.axis('off')
plt.tight_layout()
plt.show()


## 19. Reporte por clase

Precision, recall y F1 por cada clase (útil si hay desbalance).


In [ ]:
target_names = [sriesgos[i] if i < len(sriesgos) else 'Class {}'.format(i)
                for i in range(nClasses)]
print(classification_report(test_Y, predicted_classes, target_names=target_names))


## 20. Inferencia con una imagen nueva

Una sola celda (antes había dos casi iguales: se unificó).

1. (Opcional) Carga el `.h5` si reiniciaste el kernel.
2. Redimensiona a `21×28` (el tamaño del `input_shape`).
3. Normaliza `/255`.
4. `predict` → `argmax` → nombre en `sriesgos`.

Cambia `filenames` por tu(s) ruta(s).


In [ ]:
# ==============================================================================
# INFERENCIA EN VIDEO FOTOGRAMA A FOTOGRAMA (EXPLICADO FOR DUMMIES)
# ==============================================================================
# Si el kernel se reinició, descomenta las dos siguientes líneas:
# riesgo_model = load_model('modelo_caras.h5')

# 1. Cargamos el detector Haar Cascade para encontrar la cara en cada fotograma
cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_alt.xml'
detector_rostros = cv2.CascadeClassifier(cascade_path)

# 2. Abrir la cámara web (0) o puedes poner la ruta a un archivo (ej: 'video.mp4')
cap = cv2.VideoCapture(0)

print("Iniciando video... Presiona la tecla 'q' en la ventana del video para salir.")

while cap.isOpened():
    # Leemos un fotograma (frame) del video
    exito, frame = cap.read()
    if not exito:
        print("Fin del video o no se pudo acceder a la cámara.")
        break

    # Convertimos a blanco y negro solo para que el detector Haar encuentre las caras
    gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    caras_detectadas = detector_rostros.detectMultiScale(gris, scaleFactor=1.3, minNeighbors=5)

    # Por cada cara que aparezca en el fotograma:
    for (x, y, ancho, alto) in caras_detectadas:
        # A) Recortamos solo el cuadrito de la cara
        recorte_cara = frame[y:y+alto, x:x+ancho]
        
        # B) OpenCV lee en BGR, pero la CNN se entrenó en RGB, así que lo convertimos
        recorte_rgb = cv2.cvtColor(recorte_cara, cv2.COLOR_BGR2RGB)
        
        # C) Redimensionamos al tamaño exacto que espera la CNN (IMG_W, IMG_H)
        cara_ajustada = cv2.resize(recorte_rgb, (IMG_W, IMG_H))
        
        # D) Normalizamos dividiendo entre 255.0 (valores de 0.0 a 1.0)
        entrada_cnn = np.array([cara_ajustada], dtype=np.float32) / 255.0
        
        # E) La CNN predice a qué persona se parece
        predicciones = riesgo_model.predict(entrada_cnn, verbose=0)
        indice_ganador = np.argmax(predicciones[0])  # El índice con mayor probabilidad
        porcentaje = predicciones[0][indice_ganador] * 100
        nombre_persona = sriesgos[indice_ganador]

        # F) Dibujamos el recuadro verde y el nombre en la pantalla
        texto = f"{nombre_persona}: {porcentaje:.1f}%"
        cv2.rectangle(frame, (x, y), (x + ancho, y + alto), (0, 255, 0), 2)
        cv2.putText(frame, texto, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    # Mostramos el fotograma con los resultados
    cv2.imshow("Reconocimiento Facial - Fotograma a Fotograma", frame)

    # Si presionas la letra 'q', se cierra la ventana
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Liberamos la cámara y cerramos las ventanas
cap.release()
cv2.destroyAllWindows()
